# Evaluation & Application

---
## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score, accuracy_score

# PyThaiNLP for Thai language tokenization
from pythainlp.tokenize import word_tokenize

# Method A: Classical
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier

# Method B: Neural
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

# Document Search
from sklearn.metrics.pairwise import cosine_similarity

print("All imports loaded successfully.")

---
## 2. Load and Preprocess Data

In [ ]:
df = pd.read_csv('labeled_songs.csv')
print(f"Dataset loaded: {len(df)} rows")
print(f"\nLabel distribution:")
print(df['label'].value_counts())

# Extract texts and labels
texts = df['lyric'].astype(str).tolist()
labels = df['label'].tolist()

# Encode labels to integers
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(labels)
num_classes = len(np.unique(y_encoded))
print(f"\nNumber of classes: {num_classes}")
print(f"Classes: {list(label_encoder.classes_)}")

# Define a custom tokenizer using PyThaiNLP
def thai_tokenizer(text):
    return word_tokenize(text, engine='newmm')

---
## 3. Evaluation: Method A — Classical 
Precision, Recall, F1 Score** for the classical approach.

In [ ]:
print("--- Method A: Classical Approach (TF-IDF + Random Forest) ---\n")

# Feature Extraction: TF-IDF
vectorizer = TfidfVectorizer(tokenizer=thai_tokenizer, token_pattern=None)
X_tfidf = vectorizer.fit_transform(texts)

# Train-test split
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_tfidf, y_encoded, test_size=0.2, random_state=42
)

# Model: Random Forest Classifier
classical_model = RandomForestClassifier(n_estimators=100, random_state=42)
classical_model.fit(X_train_c, y_train_c)

# Prediction
y_pred_c = classical_model.predict(X_test_c)

# --- Evaluation Metrics ---
print("=" * 60)
print("METHOD A: Evaluation Results")
print("=" * 60)
print(f"\nAccuracy: {accuracy_score(y_test_c, y_pred_c):.4f}")
print(f"\nPrecision (weighted): {precision_score(y_test_c, y_pred_c, average='weighted'):.4f}")
print(f"Recall    (weighted): {recall_score(y_test_c, y_pred_c, average='weighted'):.4f}")
print(f"F1 Score  (weighted): {f1_score(y_test_c, y_pred_c, average='weighted'):.4f}")
print(f"\n--- Per-Class Classification Report ---\n")
print(classification_report(y_test_c, y_pred_c, target_names=label_encoder.classes_))

---
## 4. Evaluation: Method B — Neural
Precision, Recall, F1 Score** for the neural approach.

In [ ]:
print("--- Method B: Neural Approach (Word Embedding + LSTM) ---\n")

# Tokenize using PyThaiNLP first for all texts
tokenized_texts = [' '.join(thai_tokenizer(text)) for text in texts]

# Feature Extraction: Keras Tokenizer to create integer sequences
max_words = 5000  # Max number of words in vocabulary
max_len = 50      # Max length of each sequence

keras_tokenizer = Tokenizer(num_words=max_words)
keras_tokenizer.fit_on_texts(tokenized_texts)
sequences = keras_tokenizer.texts_to_sequences(tokenized_texts)

# Pad sequences
X_padded = pad_sequences(sequences, maxlen=max_len)

# Train-test split
X_train_n, X_test_n, y_train_n, y_test_n = train_test_split(
    X_padded, y_encoded, test_size=0.2, random_state=42
)

# One-hot encode labels for multiclass
y_train_nn = tf.keras.utils.to_categorical(y_train_n, num_classes)
y_test_nn = tf.keras.utils.to_categorical(y_test_n, num_classes)

In [ ]:
# Build LSTM Model
embedding_dim = 64

neural_model = Sequential([
    Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len),
    LSTM(64, return_sequences=False),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dense(num_classes, activation='softmax')
])

neural_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train Model
print("Training Neural Model...")
history = neural_model.fit(
    X_train_n, y_train_nn,
    epochs=10, batch_size=32,
    validation_data=(X_test_n, y_test_nn),
    verbose=1
)

loss, accuracy = neural_model.evaluate(X_test_n, y_test_nn, verbose=0)
print(f"\nNeural Model Test Accuracy: {accuracy:.4f}")

In [ ]:
# --- Evaluation Metrics for Method B ---
y_pred_prob = neural_model.predict(X_test_n)
y_pred_n = np.argmax(y_pred_prob, axis=1)

print("=" * 60)
print("METHOD B: Evaluation Results")
print("=" * 60)
print(f"\nAccuracy: {accuracy_score(y_test_n, y_pred_n):.4f}")
print(f"\nPrecision (weighted): {precision_score(y_test_n, y_pred_n, average='weighted', zero_division=0):.4f}")
print(f"Recall    (weighted): {recall_score(y_test_n, y_pred_n, average='weighted', zero_division=0):.4f}")
print(f"F1 Score  (weighted): {f1_score(y_test_n, y_pred_n, average='weighted', zero_division=0):.4f}")
print(f"\n--- Per-Class Classification Report ---\n")
print(classification_report(y_test_n, y_pred_n, target_names=label_encoder.classes_, zero_division=0))

---
## 5. Comparison: Method A vs Method B

In [ ]:
# Build comparison table
comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision (weighted)', 'Recall (weighted)', 'F1 Score (weighted)'],
    'Method A (TF-IDF + RF)': [
        accuracy_score(y_test_c, y_pred_c),
        precision_score(y_test_c, y_pred_c, average='weighted'),
        recall_score(y_test_c, y_pred_c, average='weighted'),
        f1_score(y_test_c, y_pred_c, average='weighted')
    ],
    'Method B (Embedding + LSTM)': [
        accuracy_score(y_test_n, y_pred_n),
        precision_score(y_test_n, y_pred_n, average='weighted', zero_division=0),
        recall_score(y_test_n, y_pred_n, average='weighted', zero_division=0),
        f1_score(y_test_n, y_pred_n, average='weighted', zero_division=0)
    ]
})

# Format to 4 decimal places
comparison['Method A (TF-IDF + RF)'] = comparison['Method A (TF-IDF + RF)'].map('{:.4f}'.format)
comparison['Method B (Embedding + LSTM)'] = comparison['Method B (Embedding + LSTM)'].map('{:.4f}'.format)

print("=" * 60)
print("COMPARISON: Method A vs Method B")
print("=" * 60)
print()
print(comparison.to_string(index=False))

# Document Search

In [ ]:
# Get the trained embedding weights from the neural model
embedding_layer = neural_model.layers[0]  # First layer is Embedding
embedding_weights = embedding_layer.get_weights()[0]  # shape: (max_words, embedding_dim)

print(f"Embedding matrix shape: {embedding_weights.shape}")
print(f"Vocabulary size: {embedding_weights.shape[0]}, Embedding dim: {embedding_weights.shape[1]}")

# Function to compute document vector by averaging word embeddings
def get_document_vector(sequence, embedding_matrix):
    """Average the word embeddings for all non-zero tokens in the sequence."""
    vectors = []
    for token_id in sequence:
        if token_id > 0 and token_id < len(embedding_matrix):  # skip padding (0)
            vectors.append(embedding_matrix[token_id])
    if len(vectors) == 0:
        return np.zeros(embedding_matrix.shape[1])
    return np.mean(vectors, axis=0)

# Build document vectors for all documents
print("\nBuilding document vectors...")
doc_vectors = np.array([get_document_vector(seq, embedding_weights) for seq in X_padded])
print(f"Document vectors shape: {doc_vectors.shape}")

In [ ]:
def search_documents(query, top_n=5):
    """
    Search for documents similar to the query using cosine similarity.
    
    Args:
        query (str): Thai text query
        top_n (int): Number of top results to return
    
    Returns:
        DataFrame with top matching documents
    """
    # Tokenize the query using PyThaiNLP
    query_tokenized = ' '.join(thai_tokenizer(query))
    
    # Convert to sequence using the same Keras tokenizer
    query_seq = keras_tokenizer.texts_to_sequences([query_tokenized])
    query_padded = pad_sequences(query_seq, maxlen=max_len)
    
    # Get query vector
    query_vector = get_document_vector(query_padded[0], embedding_weights).reshape(1, -1)
    
    # Compute cosine similarity against all documents
    similarities = cosine_similarity(query_vector, doc_vectors)[0]
    
    # Get top-N indices
    top_indices = np.argsort(similarities)[::-1][:top_n]
    
    # Build results
    results = []
    for rank, idx in enumerate(top_indices, 1):
        results.append({
            'Rank': rank,
            'Song & Author': df.iloc[idx]['Song & Author'],
            'Label': df.iloc[idx]['label'],
            'Similarity': f"{similarities[idx]:.4f}",
            'Lyric (preview)': df.iloc[idx]['lyric'][:100] + '...'
        })
    
    return pd.DataFrame(results)

print("Document search function created successfully.")

# Test Document Search

In [ ]:
# Test Query 1: Love song query
query1 = "รักเธอมากที่สุด อยากอยู่ด้วยกันตลอดไป"
print("=" * 60)
print(f"Query: {query1}")
print("=" * 60)
results1 = search_documents(query1, top_n=5)
print(results1.to_string(index=False))

In [ ]:
# Test Query 2: Sad song query
query2 = "เสียใจ น้ำตาไหล เจ็บปวด คิดถึง"
print("=" * 60)
print(f"Query: {query2}")
print("=" * 60)
results2 = search_documents(query2, top_n=5)
print(results2.to_string(index=False))

In [ ]:
# Test Query 3: Folk/Isan song query
query3 = "อีสาน บ้านนา ทุ่งนา คิดฮอด"
print("=" * 60)
print(f"Query: {query3}")
print("=" * 60)
results3 = search_documents(query3, top_n=5)
print(results3.to_string(index=False))

In [ ]:
# Interactive: Try your own query
custom_query = "ความรักที่ไม่สมหวัง"  # Change this to any Thai text
print("=" * 60)
print(f"Custom Query: {custom_query}")
print("=" * 60)
results_custom = search_documents(custom_query, top_n=5)
print(results_custom.to_string(index=False))